In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

## Data Visulaization

In [4]:
df=pd.read_csv('ALL_CARS_DATA.csv')
df.head()

,BRAND,MODEL/CLASS,CAR NAME,MODEL,CAR TYPE,PRICE RANGE,PRICE($),AGE OF CAR,STOCK TYPE,MILEAGE,RATING
0,Chevrolet,Traverse LT,Chevrolet Traverse LT,2024,crossover,High,41170.0,0,New,0.0,4.0
1,Chevrolet,Traverse LT,Chevrolet Traverse LT,2024,crossover,High,42540.0,0,New,0.0,4.6
2,Chevrolet,Traverse LS,Chevrolet Traverse LS,2024,crossover,High,37810.0,0,New,0.0,4.6
3,Chevrolet,Traverse 2LT,Chevrolet Traverse 2LT,2015,crossover,Low,8995.0,9,Used,156612.0,4.9
4,Chevrolet,Traverse 2LT,Chevrolet Traverse 2LT,2016,crossover,Low,12795.0,8,Used,95852.0,4.7


## EDA and Preprocessing

In [5]:
# Remove missing values properly
df = df.dropna(subset=["MODEL", "MILEAGE", "PRICE($)"])

# Remove negative values if any
df = df[df["MILEAGE"] >= 0]
df = df[df["PRICE($)"] > 0]
df = df[df["AGE OF CAR"] >= 0]
df = df[df["RATING"] >= 0]

In [6]:
# outlier handling
Q1 = df["PRICE($)"].quantile(0.25)
Q3 = df["PRICE($)"].quantile(0.75)
IQR = Q3 - Q1

df = df[(df["PRICE($)"] >= Q1 - 1.5*IQR) &
        (df["PRICE($)"] <= Q3 + 1.5*IQR)]

print("After Cleaning Shape:", df.shape)

After Cleaning Shape: (198411, 11)


In [7]:
# feature selection
X = df[["MODEL", "MILEAGE", "AGE OF CAR", 
        "RATING", "BRAND", "CAR TYPE", "STOCK TYPE"]]

y = df["PRICE($)"]

# Encode categorical variables
X = pd.get_dummies(X, drop_first=True)

In [8]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [9]:
# scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

# Evaluation
r2_lr = r2_score(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)

print("\n----- Linear Regression Results -----")
print("R2 Score:", r2_lr)
print("MSE:", mse_lr)
print("RMSE:", rmse_lr)


----- Linear Regression Results -----
R2 Score: 0.7118311269212307
MSE: 117281367.80312389
RMSE: 10829.652247561964


In [12]:
avg_mileage_per_year = df.groupby("MODEL")["MILEAGE"].mean()
print(avg_mileage_per_year)


MODEL
1919    10240.000000
1926    28300.000000
1928    32558.000000
1929    24038.200000
1930    14221.000000
            ...     
2020    51597.197012
2021    40375.602639
2022    36533.231308
2023    14048.978663
2024      359.848497
Name: MILEAGE, Length: 92, dtype: float64


In [13]:
# predict price of a car based on milage and year

price_df = df.dropna(subset=["MODEL", "MILEAGE", "PRICE($)"])

X_price = price_df[["MODEL", "MILEAGE"]]
y_price = price_df["PRICE($)"]

price_model = LinearRegression()
price_model.fit(X_price, y_price)

print("Price Prediction Model:")
print("Coefficients:", price_model.coef_)
print("Intercept:", price_model.intercept_)

Price Prediction Model:
Coefficients: [ 3.79139324e+02 -2.78815216e-01]
Intercept: -716346.451221742


In [14]:
# number of car models per year

models_per_year = df.groupby("MODEL")["CAR NAME"].nunique()
print(models_per_year)

MODEL
1919       1
1926       1
1928       1
1929       3
1930       2
        ... 
2020     801
2021     977
2022     960
2023    1140
2024    1828
Name: CAR NAME, Length: 92, dtype: int64


In [15]:
# car type per year

car_type_per_year = df.groupby(["MODEL", "CAR TYPE"]).size()
print(car_type_per_year)

MODEL  CAR TYPE     
1919   convertible          1
1926   convertible          1
1928   convertible          5
1929   Sedan                1
       convertible          4
                        ...  
2024   crossover        26147
       diesel van        5237
       hatchback         3005
       minivan           4325
       pickup-trucks     1713
Length: 656, dtype: int64


In [16]:
# car type per car name

car_type_per_name = df[["CAR NAME", "CAR TYPE"]].drop_duplicates()
print(car_type_per_name)

                                  CAR NAME   CAR TYPE
0                    Chevrolet Traverse LT  crossover
2                    Chevrolet Traverse LS  crossover
3                   Chevrolet Traverse 2LT  crossover
5            Chevrolet Traverse LT Leather  crossover
8               Chevrolet Traverse Premier  crossover
...                                    ...        ...
266392      Porsche Cayenne Cayenne (MY24)        SUV
266530    Porsche Cayenne Sport Utility 4D        SUV
266594       Porsche Cayenne SPORT UTILITY        SUV
266663                 Porsche Cayenne 4dr        SUV
266774  Porsche Cayenne AWD PANORAMIC ROOF        SUV

[7570 rows x 2 columns]
